# 03 - Analyses Thibaut Renard - Tableau vs Power BI

> Notebook de restitution analytique orienté besoins métier et choix d'outil BI.

Ce notebook traduit les indicateurs préparés dans le pipeline en analyses exploitables pour la soutenance, en distinguant les trois domaines d'expertise et les usages Tableau versus Power BI.

## Objectif et sommaire

> Objectif : transformer les jeux de données préparés en angles d'analyse lisibles pour la décision et la démonstration.

Lecture recommandée :

1. rappeler les hypothèses et limites ;
2. présenter les indicateurs nationaux à suivre ;
3. détailler les analyses des domaines création, modernisation et gouvernance ;
4. comparer les usages Tableau et Power BI ;
5. conclure sur les messages clés à porter en soutenance.

In [ ]:
import importlib
import numpy as np
import pandas as pd
import plotly.express as px

import src.schema_fr as schema_fr
importlib.reload(schema_fr)

FRENCH_COLUMN_LABELS = schema_fr.construire_libelles_analyse_fr()
VISUELS_NOTEBOOK_3_FR = schema_fr.construire_visuels_notebook_3_fr()
paths = schema_fr.resolve_project_paths()
PROJECT_ROOT = paths['PROJECT_ROOT']
PROCESSED_DATA_PATH = paths['PROCESSED_DATA_PATH']
CSV_EN_PATH = paths['CSV_EN_PATH']

def afficher_tableau_fr(dataframe, columns=None, n=None):
    table = dataframe.copy()
    if columns is not None:
        table = table[columns]
    if n is not None:
        table = table.head(n)
    return table.rename(columns=FRENCH_COLUMN_LABELS)

dashboard_water = pd.read_csv(CSV_EN_PATH / 'dashboard_water_country_year.csv', sep=';')
priority_snapshot_focus = pd.read_csv(CSV_EN_PATH / 'priority_snapshot_focus_2016.csv', sep=';')
country_region_reference = pd.read_csv(CSV_EN_PATH / 'country_region_reference.csv', sep=';')

analysis_base = dashboard_water.merge(
    country_region_reference[['country_clean', 'region']].rename(
        columns={'country_clean': 'country', 'region': 'region_reference'}
    ),
    on='country',
    how='left',
)
analysis_base['region'] = analysis_base['region'].fillna(analysis_base['region_reference'])
analysis_base = analysis_base.drop(columns=['region_reference'])

analysis_base['population_total_millions'] = analysis_base['population_total_people'] / 1_000_000
analysis_base['urban_share_pct'] = 100 - analysis_base['rural_share_pct']
analysis_base['service_quality_gap_pct'] = (
    analysis_base['basic_drinking_water_pct'] - analysis_base['safely_managed_drinking_water_pct']
)

if 'gov_effectiveness_score' not in analysis_base.columns:
    wash_series = pd.to_numeric(analysis_base['wash_mortality_rate_per_100k'], errors='coerce')
    wash_min = wash_series.min(skipna=True)
    wash_max = wash_series.max(skipna=True)
    if pd.isna(wash_min) or pd.isna(wash_max) or wash_min == wash_max:
        analysis_base['wash_score_inverse'] = pd.Series(0.0, index=analysis_base.index, dtype='float64')
    else:
        analysis_base['wash_score_inverse'] = 1 - (
            (wash_series - wash_min) / (wash_max - wash_min)
        )
    analysis_base['water_access_score'] = pd.to_numeric(
        analysis_base['basic_drinking_water_pct'], errors='coerce'
    ) / 100
    analysis_base['gov_effectiveness_score'] = (
        0.5 * analysis_base['wash_score_inverse']
        + 0.5 * analysis_base['water_access_score']
    )

analysis_base['government_effectiveness_proxy'] = analysis_base['gov_effectiveness_score']

print(f"Jointure réalisée : {len(analysis_base):,} lignes dans la base d'analyse.")
display(
    afficher_tableau_fr(
        analysis_base,
        columns=[
            'country', 'region', 'year', 'population_total_millions',
            'basic_drinking_water_pct', 'safely_managed_drinking_water_pct',
            'wash_mortality_rate_per_100k', 'gov_effectiveness_score', 'political_stability'
        ],
        n=5,
    )
)

Jointure réalisée : 4,466 lignes dans la base d'analyse.


,Pays,Région,Année,Population (millions),Accès à l'eau potable (%),Service d'eau géré en toute sécurité (%),Mortalité WASH pour 100k,Score d'effectivité gouvernementale,Stabilité politique
0,Afghanistan,Eastern Mediterranean,2000,20.779953,27.77190,NaN,NaN,NaN,-2.44
1,Afghanistan,Eastern Mediterranean,2001,21.606988,27.79726,NaN,NaN,NaN,NaN
2,Afghanistan,Eastern Mediterranean,2002,22.600770,29.90076,NaN,NaN,NaN,-2.04
3,Afghanistan,Eastern Mediterranean,2003,23.680871,32.00507,NaN,NaN,NaN,-2.20
4,Afghanistan,Eastern Mediterranean,2004,24.726684,34.12623,NaN,NaN,NaN,-2.30


In [ ]:
latest_population_year = int(analysis_base['year'].max())

population_unit_check = (
    analysis_base.loc[analysis_base['year'] == latest_population_year, [
        'country', 'population_total_people', 'population_total_millions'
    ]]
    .dropna(subset=['population_total_people'])
    .drop_duplicates(subset=['country'])
    .copy()
)

world_total_people = population_unit_check['population_total_people'].sum()
world_total_billions = world_total_people / 1_000_000_000
world_total_from_millions = population_unit_check['population_total_millions'].sum() / 1_000
conversion_gap = abs(world_total_billions - world_total_from_millions)

china_check = population_unit_check[population_unit_check['country'] == 'China'].copy()
if not china_check.empty:
    china_check['population_people_from_millions'] = china_check['population_total_millions'] * 1_000_000

population_unit_summary = pd.DataFrame(
    [
        {
            'controle': 'Somme population_total_people',
            'annee': latest_population_year,
            'valeur': round(world_total_people, 0),
            'unite': 'habitants',
        },
        {
            'controle': 'Somme population_total_people convertie',
            'annee': latest_population_year,
            'valeur': round(world_total_billions, 3),
            'unite': 'milliards',
        },
        {
            'controle': 'Somme population_total_millions reconvertie',
            'annee': latest_population_year,
            'valeur': round(world_total_from_millions, 3),
            'unite': 'milliards',
        },
        {
            'controle': 'Ecart entre les deux lectures',
            'annee': latest_population_year,
            'valeur': round(conversion_gap, 9),
            'unite': 'milliards',
        },
    ]
)

display(population_unit_summary)
if not china_check.empty:
    display(china_check)

,controle,annee,valeur,unite
0,Somme population_total_people,2018,9.090746e+09,habitants
1,Somme population_total_people convertie,2018,9.091000e+00,milliards
2,Somme population_total_millions reconvertie,2018,9.091000e+00,milliards
3,Ecart entre les deux lectures,2018,0.000000e+00,milliards


,country,population_total_people,population_total_millions,population_people_from_millions
824,China,1.459378e+09,1459.377612,1.459378e+09


### Contrôle d'unité sur la population

Ce contrôle vérifie sur une seule année que `population_total_people` est bien exprimée en habitants, et que `population_total_millions` n'est qu'une version de lecture en millions.

Il ne s'agit pas d'une conversion redondante :

. Dans le notebook 01, la source brute est d'abord remise dans son unité canonique en habitants en multipliant par `1000` ;
. Dans ce notebook, on divise ensuite par `1_000_000` pour créer une colonne plus lisible dans les graphiques ;
. Les deux colonnes décrivent donc la même population, mais avec deux unités différentes : habitants versus millions d'habitants.

Le tableau de contrôle ci-dessus montre que la somme mondiale lue en habitants puis reconvertie en milliards est identique à la somme lue via la colonne en millions. L'écart nul confirme que la chaîne de conversion est cohérente et qu'il ne s'agit pas d'un double comptage.

<a id="RNCP37837BC03"></a>

## Partie 1 - Préparation technique et indicateurs

<div style="font-size: 12px; color: #666; margin: 8px 0 4px 0;">
<strong>Bloc compétence :</strong> <span style="background-color: #F3E5F5; padding: 2px 6px; border-radius: 3px;">RNCP37837BC03</span> — Solution de visualisation · Graphiques accessibles · Reporting des tendances
</div>

> Cette partie prépare la base d'analyse, matérialise une jointure sur fichiers, définit les filtres du notebook et crée les agrégations nécessaires aux visuels demandés.

Les indicateurs demandés pendant la réunion peuvent être organisés à l'échelle nationale de la manière suivante :

1. taux de mortalité WASH pour 100 000 habitants (`wash_mortality_rate_per_100k`) ;
2. population nationale totale (`population_total_people`) ;
3. part d'habitants ayant accès à un service d'eau potable de base et à un service géré en toute sécurité (`basic_drinking_water_pct` et `safely_managed_drinking_water_pct`) ;
4. stabilité politique (`political_stability`) ;
5. évolution dans le temps : disponible surtout pour population, eau potable et stabilité politique.

Point d'interprétation important pour la soutenance :

- `wash_mortality_rate_per_100k` est un taux standardisé pour 100 000 habitants, donc une valeur décimale est normale ;
- `wash_deaths` correspond à un nombre de décès estimé par modélisation sanitaire WASH, et peut donc lui aussi rester décimal ;
- dans ce projet, ces deux variables doivent être lues comme des signaux de vulnérabilité sanitaire, et non comme des décomptes administratifs exhaustifs de l'eau potable seule.

In [ ]:
filter_year_min = 2010
filter_year_max = 2018
filter_regions = ['Africa', 'Americas', 'Eastern Mediterranean', 'Europe', 'Western Pacific']
filter_actions = [
    'Créer ou étendre les services d\'eau',
    'Moderniser les services d\'eau',
    'Renforcer la gouvernance politique',
]
filter_min_population_millions = 0.5
current_analysis_year = 2016

analysis_filtered = analysis_base[
    (analysis_base['year'] >= filter_year_min)
    & (analysis_base['year'] <= filter_year_max)
    & (analysis_base['population_total_millions'] >= filter_min_population_millions)
] .copy()

if filter_regions:
    analysis_filtered = analysis_filtered[analysis_filtered['region'].isin(filter_regions)].copy()

snapshot_filtered = analysis_filtered[analysis_filtered['year'] == current_analysis_year].copy()
if filter_actions:
    snapshot_filtered = snapshot_filtered[snapshot_filtered['action_prioritaire'].isin(filter_actions)].copy()

line_base = (
    analysis_filtered.groupby(['year', 'region'], as_index=False)
    .agg(
        avg_basic_drinking_water_pct=('basic_drinking_water_pct', 'mean'),
        avg_political_stability=('political_stability', 'mean'),
        avg_population_millions=('population_total_millions', 'mean'),
    )
)

line_metrics = line_base.melt(
    id_vars=['year', 'region'],
    value_vars=['avg_basic_drinking_water_pct', 'avg_political_stability'],
    var_name='metric',
    value_name='value',
)

filter_summary = pd.DataFrame(
    [
        {'filter_type': 'temporel', 'parameter': 'filter_year_min / filter_year_max', 'value': f'{filter_year_min} - {filter_year_max}'},
        {'filter_type': 'géographique', 'parameter': 'filter_regions', 'value': ', '.join(filter_regions)},
        {'filter_type': 'qualitatif', 'parameter': 'filter_actions', 'value': ', '.join(filter_actions)},
        {'filter_type': 'quantitatif', 'parameter': 'filter_min_population_millions', 'value': filter_min_population_millions},
    ]
)

display(afficher_tableau_fr(filter_summary))
print(f"Lignes après filtres temporel, géographique et quantitatif : {len(analysis_filtered):,}")
print(f"Pays retenus pour le snapshot {current_analysis_year} : {snapshot_filtered['country'].nunique():,}")

,Type de filtre,Paramètre,Valeur
0,temporel,filter_year_min / filter_year_max,2010 - 2018
1,géographique,filter_regions,"Africa, Americas, Eastern Mediterranean, Europ..."
2,qualitatif,filter_actions,"Créer ou étendre les services d'eau, Modernise..."
3,quantitatif,filter_min_population_millions,0.5


Lignes après filtres temporel, géographique et quantitatif : 1,460
Pays retenus pour le snapshot 2016 : 163


In [ ]:
line_visual = VISUELS_NOTEBOOK_3_FR['courbe_temporelle']

fig_line = px.line(
    line_metrics,
    x='year',
    y='value',
    color='region',
    facet_row='metric',
    markers=True,
    color_discrete_sequence=px.colors.qualitative.Safe,
    category_orders={'metric': ['avg_basic_drinking_water_pct', 'avg_political_stability']},
    labels=line_visual['labels'],
    title=line_visual['title'],
)
fig_line.for_each_annotation(
    lambda annotation: annotation.update(
        text=line_visual['facet_labels'].get(
            annotation.text.split('=')[-1],
            annotation.text.split('=')[-1],
        )
    )
)
fig_line.update_yaxes(matches=None)
fig_line.update_layout(
    template='plotly_white',
    height=720,
    legend_title_text=line_visual['legend_title'],
)
fig_line.show()

<a id="RNCP37837BC03-TEMPORAL"></a>

## Partie 2 - Lecture de la courbe temporelle

<div style="font-size: 12px; color: #666; margin: 8px 0 4px 0;">
<strong>Compétence :</strong> <span style="background-color: #F3E5F5; padding: 2px 6px; border-radius: 3px;">Analyse temporelle · Reporting des tendances</span> — Analyser les tendances pour orienter les décisions stratégiques
</div>

> Le graphique précédent porte la représentation temporelle attendue. Il montre l'évolution moyenne par région de l'accès à l'eau potable et de la stabilité politique après filtres.

<a id="RNCP37837BC03-VIZ1"></a>

## Partie 3 - Domaine 1 - Création de services

<div style="font-size: 12px; color: #666; margin: 8px 0 4px 0;">
<strong>Compétence :</strong> <span style="background-color: #F3E5F5; padding: 2px 6px; border-radius: 3px;">Créer un tableau de bord · Graphiques accessibles</span>
</div>

### Définition métier

Le besoin est d'identifier les pays où le développement de nouvelles infrastructures d'eau potable est prioritaire, en tenant compte de l'accès actuel et de la structure urbaine ou rurale de la population.

### Visualisation recommandée

Un nuage de points combinant le taux d'accès à l'eau potable et le taux de population urbaine. Cela permet d'opposer des besoins d'infrastructure urbaine dense à des besoins de desserte rurale plus diffuse.

In [ ]:
domain_creation = snapshot_filtered[[
    'country',
    'region',
    'population_total_millions',
    'basic_drinking_water_pct',
    'urban_share_pct',
    'rural_share_pct',
    'score_creation_services',
]].dropna().copy()
domain_creation = domain_creation.sort_values('score_creation_services', ascending=False).head(20)

creation_visual = VISUELS_NOTEBOOK_3_FR['creation_services']

fig_creation = px.scatter(
    domain_creation,
    x='urban_share_pct',
    y='basic_drinking_water_pct',
    size='population_total_millions',
    color='region',
    hover_name='country',
    hover_data={
        'population_total_millions': ':.2f',
        'score_creation_services': ':.1f',
        'rural_share_pct': ':.1f',
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
    labels=FRENCH_COLUMN_LABELS,
    title=creation_visual['title'],
)
fig_creation.update_layout(template='plotly_white', legend_title_text=creation_visual['legend_title'])
fig_creation.update_xaxes(range=[0, 100])
fig_creation.update_yaxes(range=[0, 100])
fig_creation.show()

display(afficher_tableau_fr(domain_creation, n=15))

,Pays,Région,Population (millions),Accès à l'eau potable (%),Part urbaine (%),Part rurale (%),Score création de services
765,Chad,Africa,14.561660,38.85259,23.256408,76.743592,80.033254
3767,South Sudan,Africa,10.832518,40.81390,8.648442,91.351558,73.638902
2901,Niger,Africa,20.788798,49.50156,16.756375,83.243625,68.454585
746,Central African Republic,Africa,4.537686,46.33376,39.872591,60.127409,66.620769
1362,Ethiopia,Africa,103.603462,40.04142,20.794171,79.205829,65.857546
3729,Somalia,Eastern Mediterranean,14.185636,50.81511,43.291869,56.708131,64.300440
1305,Eritrea,Africa,3.376557,51.84972,10.236463,89.763537,62.707321
632,Burundi,Africa,10.487995,60.20415,12.086571,87.913429,62.480822
1134,Democratic Republic of the Congo,Africa,78.789127,42.97677,43.345086,56.654914,61.988033
613,Burkina Faso,Africa,18.646357,48.26772,28.133302,71.866698,60.757072


In [ ]:
population_visual = VISUELS_NOTEBOOK_3_FR['structure_population']

population_mix = domain_creation[['country', 'urban_share_pct', 'rural_share_pct']].copy()
population_mix = population_mix.melt(
    id_vars='country',
    value_vars=['urban_share_pct', 'rural_share_pct'],
    var_name='population_split',
    value_name='share_pct',
)
population_mix['population_split'] = population_mix['population_split'].replace(population_visual['split_labels'])

fig_population_mix = px.bar(
    population_mix,
    x='country',
    y='share_pct',
    color='population_split',
    barmode='relative',
    color_discrete_sequence=px.colors.qualitative.Safe,
    labels=population_visual['labels'],
    title=population_visual['title'],
)
fig_population_mix.update_layout(template='plotly_white', legend_title_text=population_visual['legend_title'])
fig_population_mix.update_yaxes(range=[0, 100])
fig_population_mix.show()

### Lecture du graphique en barres empilées en pourcentage

> Le graphique précédent complète le nuage de points en montrant la structure relative urbaine et rurale des pays les plus prioritaires dans la création de services.

### Conclusion du domaine 1

Le couple `basic_drinking_water_pct` et `urban_share_pct` permet de distinguer les pays à faible accès dans des contextes urbains de ceux où la dispersion rurale complique la création de services. Le score de création sert à prioriser, mais la lecture du nuage de points sert à comprendre la nature de l'effort à fournir.

## Partie 4 - Domaine 2 - Modernisation des services

### Définition métier

Le besoin est d'identifier les pays où un service de base existe déjà, mais où la qualité du service reste insuffisante. L'écart entre les services de base et les services gérés en toute sécurité devient alors un indicateur central.

### Visualisation recommandée

Un graphique en barres groupées permet de comparer directement le taux de services de base et le taux de services gérés en toute sécurité pour les pays les plus prioritaires.

In [ ]:
domain_modernisation = snapshot_filtered[[
    'country',
    'region',
    'basic_drinking_water_pct',
    'safely_managed_drinking_water_pct',
    'service_quality_gap_pct',
    'score_modernisation_services',
]].dropna().copy()
domain_modernisation = domain_modernisation.sort_values('score_modernisation_services', ascending=False).head(12)

modernisation_visual = VISUELS_NOTEBOOK_3_FR['modernisation_services']

modernisation_long = domain_modernisation.melt(
    id_vars=['country', 'region', 'score_modernisation_services'],
    value_vars=['basic_drinking_water_pct', 'safely_managed_drinking_water_pct'],
    var_name='service_level',
    value_name='coverage_pct',
)
modernisation_long['service_level'] = modernisation_long['service_level'].replace(modernisation_visual['service_level_labels'])

fig_modernisation = px.bar(
    modernisation_long,
    x='country',
    y='coverage_pct',
    color='service_level',
    barmode='group',
    color_discrete_sequence=px.colors.qualitative.Safe,
    labels={**FRENCH_COLUMN_LABELS, **modernisation_visual['labels']},
    title=modernisation_visual['title'],
)
fig_modernisation.update_layout(template='plotly_white', legend_title_text=modernisation_visual['legend_title'])
fig_modernisation.update_yaxes(range=[0, 100])
fig_modernisation.show()

display(afficher_tableau_fr(domain_modernisation, n=12))

,Pays,Région,Accès à l'eau potable (%),Service d'eau géré en toute sécurité (%),Écart de qualité de service,Score modernisation des services
3626,Sierra Leone,Africa,59.56314,9.53397,50.02917,85.269905
2920,Nigeria,Africa,70.02553,19.90354,50.12199,77.924590
2198,Lao People's Democratic Republic,Western Pacific,79.94190,15.68307,64.25883,76.067516
4159,Uganda,Africa,47.66563,6.95153,40.71410,71.701163
2635,Mongolia,Western Pacific,82.80058,23.61521,59.18537,67.653216
1362,Ethiopia,Africa,40.04142,10.86826,29.17316,66.129143
670,Cambodia,Western Pacific,76.94537,25.24514,51.70023,63.940050
3015,Pakistan,Eastern Mediterranean,91.14014,35.53562,55.60452,63.757607
1590,Ghana,Africa,80.43578,34.75200,45.68378,58.715239
1096,Côte d'Ivoire,Africa,72.76205,36.39219,36.36986,58.607706


### Conclusion du domaine 2

Le meilleur graphique est une combinaison entre le taux de service de base et le taux de service géré en toute sécurité. Plus l'écart est élevé, plus le besoin porte sur l'amélioration qualitative des services existants, et non sur une création ex nihilo.

<a id="RNCP37837BC03-VIZ3"></a>

## Partie 5 - Domaine 3 - Consulting et gouvernance

<div style="font-size: 12px; color: #666; margin: 8px 0 4px 0;">
<strong>Compétences :</strong> <span style="background-color: #F3E5F5; padding: 2px 6px; border-radius: 3px;">Récit des résultats · Présenter les résultats</span> — Adapter le contenu au public pour transmettre les informations clés
</div>

### Définition métier

Le besoin est d'approcher des pays où le consulting public sur l'eau a du sens, en combinant la performance observable de la politique d'accès à l'eau et le niveau de stabilité politique.

### Visualisation recommandée

Un nuage de points à quadrants permet de croiser l'efficacité observable de la politique d'accès à l'eau et la stabilité politique, tout en gardant une lecture adaptée à un auditoire métier.

In [ ]:
domain_governance = snapshot_filtered[[
    'country',
    'region',
    'population_total_millions',
    'political_stability',
    'basic_drinking_water_pct',
    'wash_mortality_rate_per_100k',
    'gov_effectiveness_score',
    'score_gouvernance',
    'action_prioritaire',
]].dropna().copy()
domain_governance = domain_governance.sort_values('score_gouvernance', ascending=False).head(20)

governance_visual = VISUELS_NOTEBOOK_3_FR['gouvernance']

fig_governance = px.scatter(
    domain_governance,
    x='political_stability',
    y='gov_effectiveness_score',
    size='population_total_millions',
    color='region',
    symbol='action_prioritaire',
    hover_name='country',
    hover_data={
        'score_gouvernance': ':.1f',
        'gov_effectiveness_score': ':.2f',
        'wash_mortality_rate_per_100k': ':.1f',
        'basic_drinking_water_pct': ':.1f',
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
    labels=FRENCH_COLUMN_LABELS,
    title=governance_visual['title'],
)
fig_governance.add_vline(
    x=domain_governance['political_stability'].median(),
    line_dash='dash',
    line_color='gray',
)
fig_governance.add_hline(
    y=domain_governance['gov_effectiveness_score'].median(),
    line_dash='dash',
    line_color='gray',
)
fig_governance.update_layout(template='plotly_white', legend_title_text=governance_visual['legend_title'])
fig_governance.show()

display(afficher_tableau_fr(domain_governance, n=15))

,Pays,Région,Population (millions),Stabilité politique,Accès à l'eau potable (%),Mortalité WASH pour 100k,Score d'effectivité gouvernementale,Score gouvernance,Action prioritaire
2160,Kuwait,Eastern Mediterranean,3.956875,-0.05,100.00000,0.01242,0.999970,99.996957,Renforcer la gouvernance politique
1438,Finland,Europe,5.497713,1.00,100.00000,0.03616,0.999852,99.985208,Renforcer la gouvernance politique
1628,Greece,Europe,10.615185,-0.12,100.00000,0.03710,0.999847,99.984743,Renforcer la gouvernance politique
301,Bahrain,Eastern Mediterranean,1.425792,-0.79,100.00000,0.03960,0.999835,99.983506,Renforcer la gouvernance politique
3645,Singapore,Western Pacific,5.653634,1.50,100.00000,0.06611,0.999704,99.970387,Renforcer la gouvernance politique
244,Austria,Europe,8.747301,0.91,100.00000,0.12375,0.999419,99.941863,Créer ou étendre les services d'eau
225,Australia,Western Pacific,24.262712,1.05,99.96997,0.09539,0.999409,99.940882,Renforcer la gouvernance politique
3893,Switzerland,Europe,8.379917,1.31,100.00000,0.13784,0.999349,99.934890,Créer ou étendre les services d'eau
2350,Luxembourg,Europe,0.579264,1.42,99.90687,0.04850,0.999325,99.932537,Renforcer la gouvernance politique
2863,New Zealand,Western Pacific,4.659265,1.52,100.00000,0.14577,0.999310,99.930966,Renforcer la gouvernance politique


In [ ]:
map_base = snapshot_filtered[[
    'country',
    'region',
    'score_gouvernance',
    'basic_drinking_water_pct',
    'wash_mortality_rate_per_100k',
]].dropna(subset=['score_gouvernance']).copy()

map_visual = VISUELS_NOTEBOOK_3_FR['carte_gouvernance']

fig_map = px.choropleth(
    map_base,
    locations='country',
    locationmode='country names',
    color='score_gouvernance',
    hover_name='country',
    hover_data={
        'region': True,
        'basic_drinking_water_pct': ':.1f',
        'wash_mortality_rate_per_100k': ':.1f',
        'score_gouvernance': ':.1f',
    },
    color_continuous_scale='YlOrRd',
    labels=FRENCH_COLUMN_LABELS,
    title=map_visual['title'],
)
fig_map.update_geos(showcoastlines=True, showframe=False)
fig_map.update_layout(template='plotly_white')
fig_map.show()

### Lecture de la carte choroplèthe

> La carte précédente fournit la représentation géographique demandée et localise le score de gouvernance sur le snapshot 2016.

### Conclusion du domaine 3

Le graphique recommandé combine une mesure d'efficacité observable, ici approximée par un bon accès à l'eau et une faible mortalité WASH, avec la stabilité politique. Le résultat n'est pas une preuve causale, mais un cadrage utile pour la priorisation du consulting.

### Justification méthodologique du score de gouvernance

Le score d'effectivité gouvernementale utilisé ici repose sur une logique plus défendable que l'ancien proxy par soustraction directe.

1. `wash_mortality_rate_per_100k` et `basic_drinking_water_pct` n'ont pas la même échelle ; une soustraction brute mélangeait donc deux ordres de grandeur incomparables.
2. La normalisation min-max de la mortalité WASH ramène ce signal sur une échelle `0-1`, puis son inversion permet de lire les meilleures situations vers le haut.
3. Le taux d'accès à l'eau est lui aussi converti sur `0-1`, ce qui rend la moyenne pondérée `50/50` interprétable.
4. `score_gouvernance` reste ensuite exprimé sur `0-100` pour conserver une continuité avec les logiques de priorisation déjà utilisées dans le projet.

Ce score ne prétend pas mesurer causalement la qualité d'un gouvernement. Il sert à construire un indicateur composite lisible pour la priorisation métier et pour la restitution Power BI.

## Partie 6 - Traduction opérationnelle dans Power BI

### Cadrage de la page "Domaines métiers"

Pour Power BI, la page "Domaines métiers" ne doit pas juxtaposer des graphiques indépendants. Elle doit réunir sur une même page les trois domaines demandés : création de services, modernisation des services et gouvernance, avec des slicers communs et un même périmètre de lecture.

### Pourquoi Power BI ici

Power BI est pertinent dans ce projet parce que le modèle `pbi_star` fournit déjà une table de faits exploitable et des dimensions de filtrage stables. L'enjeu n'est donc pas de réinventer les calculs, mais de traduire les visuels du notebook en composants Power BI compatibles avec ce modèle.

### Lecture recommandée

1. utiliser `fact_dashboard_star_fr` comme table principale pour la page ;
2. brancher les slicers sur l'année, la région et l'action prioritaire ;
3. réserver la page aux trois visuels métiers et laisser les vues temporelles ou qualité sur d'autres pages ;
4. appliquer les top N par score métier pour garder une page lisible en soutenance.

In [ ]:
import pandas as pd

power_bi_page_domaines_metiers = pd.DataFrame(
    [
        {
            'page': 'Page 2 - Domaines metiers',
            'domaine': 'Creation de services',
            'visuel_power_bi': 'Nuage de points',
            'table_source': 'fact_dashboard_star_fr',
            'axe_x': 'Mesure Part urbaine (%)',
            'axe_y': 'Moyenne acces_eau_potable_pct',
            'taille': 'Mesure Population (M)',
            'details_ou_legende': 'Details = pays ; Legende = region',
            'filtres_visuels': 'annee = 2016 ; Top N 20 par Score creation max ; action_prioritaire = Creer ou etendre les services d eau',
            'usage_metier': 'identifier les pays a faible acces avec forte ou faible urbanisation',
        },
        {
            'page': 'Page 2 - Domaines metiers',
            'domaine': 'Modernisation des services',
            'visuel_power_bi': 'Histogramme groupe',
            'table_source': 'fact_dashboard_star_fr',
            'axe_x': 'pays',
            'axe_y': 'Moyenne acces_eau_potable_pct et Moyenne service_safely_managed_pct',
            'taille': 'non applicable',
            'details_ou_legende': 'Legende = mesure ; Tri = Score modernisation max decroissant',
            'filtres_visuels': 'annee = 2016 ; Top N 12 par Score modernisation max ; action_prioritaire = Moderniser les services d eau',
            'usage_metier': 'rendre visible l ecart entre service de base et service gere en toute securite',
        },
        {
            'page': 'Page 2 - Domaines metiers',
            'domaine': 'Gouvernance',
            'visuel_power_bi': 'Nuage de points',
            'table_source': 'fact_dashboard_star_fr',
            'axe_x': 'Moyenne stabilite_politique',
            'axe_y': 'Moyenne score_effectivite_gouvernementale',
            'taille': 'Mesure Population (M)',
            'details_ou_legende': 'Details = pays ; Legende = region ; action_prioritaire en info-bulle',
            'filtres_visuels': 'annee = 2016 ; Top N 20 par Score gouvernance max ; action_prioritaire = Renforcer la gouvernance politique',
            'usage_metier': 'cibler les pays ou le consulting public peut etre priorise',
        },
    ]
)

power_bi_slicers = pd.DataFrame(
    [
        {'slicer': 'annee', 'source': 'dim_annee_star_fr[annee]', 'reglage_recommande': 'valeur par defaut = 2016'},
        {'slicer': 'region', 'source': 'dim_pays_star_fr[region]', 'reglage_recommande': 'multi-selection autorisee'},
        {'slicer': 'action prioritaire', 'source': 'dim_action_star_fr[action_prioritaire]', 'reglage_recommande': 'masquer Non classe sur la page metier'},
    ]
)

power_bi_measures = pd.DataFrame(
    [
        {
            'mesure': 'Population (M)',
            'dax': 'Population (M) = DIVIDE(SUM(fact_dashboard_star_fr[population_totale]), 1000000)',
            'usage': 'taille des bulles sur creation et gouvernance',
        },
        {
            'mesure': 'Part urbaine (%)',
            'dax': 'Part urbaine (%) = 100 - AVERAGE(fact_dashboard_star_fr[part_rurale_pct])',
            'usage': 'axe X du nuage Creation de services',
        },
        {
            'mesure': 'Score effectivite gouvernementale',
            'dax': 'Score effectivite gouvernementale = AVERAGE(fact_dashboard_star_fr[score_effectivite_gouvernementale])',
            'usage': 'axe Y du nuage Gouvernance si le score est deja publie dans le fact',
        },
        {
            'mesure': 'Score creation max',
            'dax': 'Score creation max = MAX(fact_dashboard_star_fr[score_creation_services])',
            'usage': 'Top N du visuel Creation de services',
        },
        {
            'mesure': 'Score modernisation max',
            'dax': 'Score modernisation max = MAX(fact_dashboard_star_fr[score_modernisation_services])',
            'usage': 'Top N du visuel Modernisation des services',
        },
        {
            'mesure': 'Score gouvernance max',
            'dax': 'Score gouvernance max = MAX(fact_dashboard_star_fr[score_gouvernance])',
            'usage': 'Top N du visuel Gouvernance',
        },
    ]
)

power_bi_dax_exact = pd.DataFrame(
    [
        {
            'mesure': 'Mortalite WASH min globale',
            'dax': """Mortalite WASH min globale =
CALCULATE(
    MIN(fact_dashboard_star_fr[mortalite_wash_pour_100k]),
    ALL(fact_dashboard_star_fr)
)""",
            'role': 'reproduire le min global utilise par le pipeline pour la normalisation',
        },
        {
            'mesure': 'Mortalite WASH max globale',
            'dax': """Mortalite WASH max globale =
CALCULATE(
    MAX(fact_dashboard_star_fr[mortalite_wash_pour_100k]),
    ALL(fact_dashboard_star_fr)
)""",
            'role': 'reproduire le max global utilise par le pipeline pour la normalisation',
        },
        {
            'mesure': 'Score mortalite WASH inverse',
            'dax': """Score mortalite WASH inverse =
VAR CurrentValue = AVERAGE(fact_dashboard_star_fr[mortalite_wash_pour_100k])
VAR MinValue = [Mortalite WASH min globale]
VAR MaxValue = [Mortalite WASH max globale]
RETURN
IF(
    ISBLANK(CurrentValue),
    BLANK(),
    IF(
        MaxValue = MinValue,
        0,
        1 - DIVIDE(CurrentValue - MinValue, MaxValue - MinValue)
    )
)""",
            'role': 'equivalent DAX de wash_score_inverse',
        },
        {
            'mesure': 'Score acces eau',
            'dax': """Score acces eau =
DIVIDE(
    AVERAGE(fact_dashboard_star_fr[acces_eau_potable_pct]),
    100
)""",
            'role': 'equivalent DAX de water_access_score',
        },
        {
            'mesure': 'Score effectivite gouvernementale exacte',
            'dax': """Score effectivite gouvernementale exacte =
0.5 * [Score mortalite WASH inverse]
    + 0.5 * [Score acces eau]""",
            'role': 'reproduction exacte du score calcule dans le pipeline',
        },
        {
            'mesure': 'Score gouvernance exact',
            'dax': """Score gouvernance exact =
100 * [Score effectivite gouvernementale exacte]""",
            'role': 'version 0-100 conservee pour les tris et top N',
        },
    ]
)

power_bi_layout = pd.DataFrame(
    [
        {'zone_page': 'haut gauche', 'contenu': 'Slicer annee', 'objectif': 'figer le snapshot de lecture'},
        {'zone_page': 'haut centre', 'contenu': 'Slicer region', 'objectif': 'filtrer sans casser les comparaisons'},
        {'zone_page': 'haut droite', 'contenu': 'Slicer action prioritaire', 'objectif': 'naviguer entre domaines si besoin'},
        {'zone_page': 'ligne 2 gauche', 'contenu': 'Nuage Creation de services', 'objectif': 'montrer le besoin de creation'},
        {'zone_page': 'ligne 2 centre-droite', 'contenu': 'Histogramme groupe Modernisation', 'objectif': 'montrer l ecart qualitatif'},
        {'zone_page': 'ligne 3 pleine largeur', 'contenu': 'Nuage Gouvernance', 'objectif': 'montrer les contextes de consulting'},
    ]
)

display(power_bi_page_domaines_metiers)
display(power_bi_slicers)
display(power_bi_measures)
display(power_bi_dax_exact)
display(power_bi_layout)

,page,domaine,visuel_power_bi,table_source,axe_x,axe_y,taille,details_ou_legende,filtres_visuels,usage_metier
0,Page 2 - Domaines metiers,Creation de services,Nuage de points,fact_dashboard_star_fr,Mesure Part urbaine (%),Moyenne acces_eau_potable_pct,Mesure Population (M),Details = pays ; Legende = region,annee = 2016 ; Top N 20 par Score creation max...,identifier les pays a faible acces avec forte ...
1,Page 2 - Domaines metiers,Modernisation des services,Histogramme groupe,fact_dashboard_star_fr,pays,Moyenne acces_eau_potable_pct et Moyenne servi...,non applicable,Legende = mesure ; Tri = Score modernisation m...,annee = 2016 ; Top N 12 par Score modernisatio...,rendre visible l ecart entre service de base e...
2,Page 2 - Domaines metiers,Gouvernance,Nuage de points,fact_dashboard_star_fr,Moyenne stabilite_politique,Moyenne score_effectivite_gouvernementale,Mesure Population (M),Details = pays ; Legende = region ; action_pri...,annee = 2016 ; Top N 20 par Score gouvernance ...,cibler les pays ou le consulting public peut e...


,slicer,source,reglage_recommande
0,annee,dim_annee_star_fr[annee],valeur par defaut = 2016
1,region,dim_pays_star_fr[region],multi-selection autorisee
2,action prioritaire,dim_action_star_fr[action_prioritaire],masquer Non classe sur la page metier


,mesure,dax,usage
0,Population (M),Population (M) = DIVIDE(SUM(fact_dashboard_sta...,taille des bulles sur creation et gouvernance
1,Part urbaine (%),Part urbaine (%) = 100 - AVERAGE(fact_dashboar...,axe X du nuage Creation de services
2,Score effectivite gouvernementale,Score effectivite gouvernementale = AVERAGE(fa...,axe Y du nuage Gouvernance si le score est dej...
3,Score creation max,Score creation max = MAX(fact_dashboard_star_f...,Top N du visuel Creation de services
4,Score modernisation max,Score modernisation max = MAX(fact_dashboard_s...,Top N du visuel Modernisation des services
5,Score gouvernance max,Score gouvernance max = MAX(fact_dashboard_sta...,Top N du visuel Gouvernance


,mesure,dax,role
0,Mortalite WASH min globale,Mortalite WASH min globale =\nCALCULATE(\n ...,reproduire le min global utilise par le pipeli...
1,Mortalite WASH max globale,Mortalite WASH max globale =\nCALCULATE(\n ...,reproduire le max global utilise par le pipeli...
2,Score mortalite WASH inverse,Score mortalite WASH inverse =\nVAR CurrentVal...,equivalent DAX de wash_score_inverse
3,Score acces eau,Score acces eau =\nDIVIDE(\n AVERAGE(fact_d...,equivalent DAX de water_access_score
4,Score effectivite gouvernementale exacte,Score effectivite gouvernementale exacte =\n0....,reproduction exacte du score calcule dans le p...
5,Score gouvernance exact,Score gouvernance exact =\n100 * [Score effect...,version 0-100 conservee pour les tris et top N


,zone_page,contenu,objectif
0,haut gauche,Slicer annee,figer le snapshot de lecture
1,haut centre,Slicer region,filtrer sans casser les comparaisons
2,haut droite,Slicer action prioritaire,naviguer entre domaines si besoin
3,ligne 2 gauche,Nuage Creation de services,montrer le besoin de creation
4,ligne 2 centre-droite,Histogramme groupe Modernisation,montrer l ecart qualitatif
5,ligne 3 pleine largeur,Nuage Gouvernance,montrer les contextes de consulting


<a id="RNCP37837BC03-CONCLUSION"></a>

## Conclusion et messages clés pour la soutenance

<div style="font-size: 12px; color: #666; margin: 8px 0 4px 0;">
<strong>Compétence :</strong> <span style="background-color: #F3E5F5; padding: 2px 6px; border-radius: 3px;">Récit des résultats · Présenter les résultats</span>
</div>

## Conclusion générale

Les trois domaines d'expertise peuvent être alimentés avec des indicateurs nationaux cohérents, à condition de bien distinguer les séries temporelles des snapshots. Le point de vigilance majeur reste la mortalité WASH, qui doit servir de référence 2016 pour la priorisation et non de preuve d'évolution. Cette clarification rend l'analyse plus crédible, quel que soit l'outil de restitution choisi.

Pour la lecture des valeurs, `wash_mortality_rate_per_100k` reste logiquement décimal car il s'agit d'un taux standardisé, tandis que `wash_deaths` peut conserver des décimales car la source fournit une estimation statistique de charge sanitaire et non un comptage exhaustif de décès.